# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toBESkiii/FlyRank_AI_Intership/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Growing vs Declining Content

The paper reports that growing pages were younger on average than declining pages. Growing pages had an average age of 185 days, compared with 228 days for declining pages, while average word count was very similar between the two groups.

**Methodology question:**
How much of the observed age difference remains after accounting for other factors such as client, topic, content type, publication cohort or seasonality?

I would ask this because the comparison shows an observed association between age and trend direction, but the groups may differ in other ways as well. A grouped or adjusted analysis could help determine whether age remains an important directional signal across different clients and types of content.

I would therefore interpret this finding as evidence that older content was **associated with** decline more often in the measured dataset, rather than evidence that age itself causes content to decline.

### Finding 4 — The Freshness Multiplier

The paper reports strong measured differences between refreshed and stale content. The 31–90 day freshness window showed a 5.43:1 growth-to-decline ratio. In a separate comparison among pages older than one year, recently refreshed pages showed substantially higher health and impressions than older stale pages.

**Methodology question:**
Were the refreshed and stale pages comparable before the refresh took place, particularly in their previous impressions, rankings, content quality and business importance?

I would ask this because pages selected for refreshing may already differ from pages that are left untouched. For example, editors may be more likely to refresh historically successful or strategically important pages. Comparing similar pages before and after refresh, or matching refreshed pages with comparable untouched pages, would provide stronger evidence about the effect associated with refreshing.

I would therefore interpret the reported refresh differences as a strong **observed and directional signal**, while avoiding the stronger claim that refreshing alone caused the measured improvement.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)


To test how sensitive my Week-5 result is to validation design, I will compare two evaluation approaches using the same Random Forest model, features, target and ranking metrics.

The first approach is a **random page-level split**. This is a weaker validation design for my use case because pages belonging to the same client may appear in both training and testing. If clients have distinctive content or search-performance patterns, this could make the test set more similar to the training data than a genuinely new client would be.

The second approach is a **grouped client-level split**. All pages belonging to one client are kept entirely in either training or testing. This provides a stronger test of whether the model can generalise its ranking to clients it did not observe during training.

I will compare both approaches using Precision@10 and Precision@50. The purpose is not to make the grouped result look better, but to measure how much the validation design affects the observed model performance.


In [2]:
import pandas as pd
import numpy as np

# Load the same starter dataset used in Week 5
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "toBESkiii/FlyRank_AI_Intership/main/"
    "data/raw/content_refresh_anonymized.csv"
)

page_dataset = pd.read_csv(DATA_URL)

# Recreate the same decline proxy used in Week 5
page_dataset["decline_proxy"] = (
    page_dataset["trend_direction"] == "down"
).astype(int)

print("Dataset loaded successfully.")
print("Dataset shape:", page_dataset.shape)
print("Number of clients:", page_dataset["client_id"].nunique())
print(
    "Decline proxy rate:",
    round(page_dataset["decline_proxy"].mean(), 3)
)

Dataset loaded successfully.
Dataset shape: (30000, 45)
Number of clients: 32
Decline proxy rate: 0.542


In [3]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np


# ---------------------------------------------------------
# 1. Use the same features and target as Week 5
# ---------------------------------------------------------

model_features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "search_volume"
]

target_column = "decline_proxy"


# ---------------------------------------------------------
# 2. Helper function to build the SAME Random Forest
# ---------------------------------------------------------

def build_random_forest():
    """
    Return the same Random Forest pipeline used in Week 5.
    """

    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=8,
                    min_samples_leaf=10,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]
    )


# ---------------------------------------------------------
# 3. Helper function for Precision@K
# ---------------------------------------------------------

def precision_at_k(y_true, ranking_scores, k):

    y_true_array = np.asarray(y_true)
    ranking_scores_array = np.asarray(ranking_scores)

    actual_k = min(k, len(y_true_array))

    top_indices = np.argsort(
        ranking_scores_array
    )[::-1][:actual_k]

    return y_true_array[top_indices].mean()


# ---------------------------------------------------------
# 4. Prepare modelling features
# ---------------------------------------------------------

model_data = page_dataset.copy()

# avg_position = 0 represents unavailable position data,
# so treat it as missing.
model_data["avg_position"] = (
    model_data["avg_position"]
    .replace(0, np.nan)
)

X = model_data[model_features].copy()
y = model_data[target_column].copy()


# =========================================================
# BEFORE: RANDOM PAGE-LEVEL SPLIT
# =========================================================

X_train_random, X_test_random, y_train_random, y_test_random, \
clients_train_random, clients_test_random = train_test_split(
    X,
    y,
    model_data["client_id"],
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_split_model = build_random_forest()

random_split_model.fit(
    X_train_random,
    y_train_random
)

random_split_scores = (
    random_split_model
    .predict_proba(X_test_random)[:, 1]
)

random_precision_10 = precision_at_k(
    y_test_random,
    random_split_scores,
    10
)

random_precision_50 = precision_at_k(
    y_test_random,
    random_split_scores,
    50
)

random_shared_clients = set(
    clients_train_random
).intersection(
    set(clients_test_random)
)


# =========================================================
# AFTER: GROUPED CLIENT-LEVEL SPLIT
# =========================================================

grouped_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train_indices, group_test_indices = next(
    grouped_split.split(
        X,
        y=y,
        groups=model_data["client_id"]
    )
)

X_train_grouped = X.iloc[group_train_indices].copy()
X_test_grouped = X.iloc[group_test_indices].copy()

y_train_grouped = y.iloc[group_train_indices].copy()
y_test_grouped = y.iloc[group_test_indices].copy()

clients_train_grouped = (
    model_data.iloc[group_train_indices]["client_id"]
)

clients_test_grouped = (
    model_data.iloc[group_test_indices]["client_id"]
)

grouped_split_model = build_random_forest()

grouped_split_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_split_scores = (
    grouped_split_model
    .predict_proba(X_test_grouped)[:, 1]
)

grouped_precision_10 = precision_at_k(
    y_test_grouped,
    grouped_split_scores,
    10
)

grouped_precision_50 = precision_at_k(
    y_test_grouped,
    grouped_split_scores,
    50
)

grouped_shared_clients = set(
    clients_train_grouped
).intersection(
    set(clients_test_grouped)
)


# ---------------------------------------------------------
# 5. Before / after comparison
# ---------------------------------------------------------

validation_comparison = pd.DataFrame(
    {
        "Validation design": [
            "Random page split",
            "Grouped client split"
        ],
        "Train pages": [
            len(X_train_random),
            len(X_train_grouped)
        ],
        "Test pages": [
            len(X_test_random),
            len(X_test_grouped)
        ],
        "Shared clients": [
            len(random_shared_clients),
            len(grouped_shared_clients)
        ],
        "Test base rate": [
            y_test_random.mean(),
            y_test_grouped.mean()
        ],
        "Precision@10": [
            random_precision_10,
            grouped_precision_10
        ],
        "Precision@50": [
            random_precision_50,
            grouped_precision_50
        ]
    }
)

validation_comparison[
    [
        "Test base rate",
        "Precision@10",
        "Precision@50"
    ]
] = validation_comparison[
    [
        "Test base rate",
        "Precision@10",
        "Precision@50"
    ]
].round(3)

print("BEFORE / AFTER VALIDATION COMPARISON")

display(validation_comparison)

print(
    "\nClients shared between train and test "
    "under random split:",
    len(random_shared_clients)
)

print(
    "Clients shared between train and test "
    "under grouped split:",
    len(grouped_shared_clients)
)

BEFORE / AFTER VALIDATION COMPARISON


,Validation design,Train pages,Test pages,Shared clients,Test base rate,Precision@10,Precision@50
0,Random page split,24000,6000,31,0.542,0.8,0.92
1,Grouped client split,23837,6163,0,0.511,0.8,0.78



Clients shared between train and test under random split: 31
Clients shared between train and test under grouped split: 0


### Before/after interpretation

The random page-level split produced Precision@10 of 0.80 and Precision@50 of 0.92. However, 31 clients appeared in both the training and test sets under this validation design.

When I changed to a grouped client-level split, there were zero shared clients between training and testing. Precision@10 remained 0.80, while Precision@50 decreased from 0.92 to 0.78.

The unchanged Precision@10 suggests that the model's very highest-ranked recommendations remained strong under the harder validation design. However, the 0.14 reduction in Precision@50 shows that the random page split gave a more optimistic picture of performance deeper in the ranked queue.

The test base rate also changed from 0.542 under the random split to 0.511 under the grouped split because entire clients, rather than individual pages, were held out.

For this use case, I therefore treat the grouped-client result as the more trustworthy estimate because it measures performance on clients that were not observed during training. The measured Precision@50 of 0.78 should be interpreted as evidence from this held-out client split rather than as a guarantee of performance on every future client.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
## 3. Leakage audit

I audited the features used by the Week-5 Random Forest to check whether any feature directly contains or derives from the proxy target.

The model uses:

* `days_since_last_update`
* `impressions_90d`
* `ctr`
* `avg_position`
* `search_volume`

The following columns are explicitly excluded from model inputs:

* `trend_direction`
* `trend_pct`
* `decline_proxy`
* `client_id`
* `content_id`

`trend_direction` and `trend_pct` are excluded because they directly define or contribute to the decline proxy and would create target leakage. Client and content identifiers are retained only for grouping, splitting and review rather than as predictive inputs.

There is also a more subtle temporal limitation. Several search-performance features, including impressions, CTR and average position, describe recent or trailing performance and may overlap with the measurement period used to construct the observed decline proxy.

Therefore, although the model does not contain an obvious label-derived feature, I should not interpret its score as a clean prediction of future decline. The model is better understood as ranking pages whose currently observed search-performance characteristics are associated with the measured decline proxy.


In [4]:
# ---------------------------------------------------------
# Leakage audit
# ---------------------------------------------------------

model_features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "search_volume"
]

direct_leakage_columns = {
    "trend_direction",
    "trend_pct",
    "decline_proxy"
}

identifier_columns = {
    "client_id",
    "content_id"
}

direct_leakage_found = [
    feature
    for feature in model_features
    if feature in direct_leakage_columns
]

identifiers_found = [
    feature
    for feature in model_features
    if feature in identifier_columns
]

leakage_audit = pd.DataFrame(
    {
        "Feature": model_features,
        "Used by model": ["Yes"] * len(model_features),
        "Direct target leakage": ["No"] * len(model_features),
        "Audit note": [
            "Freshness signal; not derived from decline proxy.",
            "Recent performance feature; temporal overlap with proxy window should be considered.",
            "Recent performance feature; temporal overlap with proxy window should be considered.",
            "Recent search-position feature; temporal overlap with proxy window should be considered.",
            "Keyword demand feature; not directly derived from decline proxy."
        ]
    }
)

print("Direct leakage features accidentally included:",
      direct_leakage_found)

print("Identifier features accidentally included:",
      identifiers_found)

display(leakage_audit)

Direct leakage features accidentally included: []
Identifier features accidentally included: []


,Feature,Used by model,Direct target leakage,Audit note
0,days_since_last_update,Yes,No,Freshness signal; not derived from decline proxy.
1,impressions_90d,Yes,No,Recent performance feature; temporal overlap w...
2,ctr,Yes,No,Recent performance feature; temporal overlap w...
3,avg_position,Yes,No,Recent search-position feature; temporal overl...
4,search_volume,Yes,No,Keyword demand feature; not directly derived f...


## 4. Claim rewrite

## 4. Claim rewrite

The Week-5 model produced strong ranking results, but the validation audit shows that the claims should remain limited to what was actually measured.

### Claim 1 — Model performance

**Too strong:**
Random Forest predicts which pages will decline.

**Rewritten claim:**
On the grouped client-held-out starter dataset, Random Forest ranked pages associated with the observed decline proxy more effectively than the Week-4 baseline. It achieved a measured Precision@10 of 0.80 and Precision@50 of 0.78.

This wording is safer because the target represents an observed recent decline state rather than a clean future outcome.

### Claim 2 — Generalisation

**Too strong:**
The model will achieve approximately 78% precision for new clients.

**Rewritten claim:**
Under one grouped client-level holdout, the model achieved Precision@50 of 0.78 on clients that were not included in training.

The random page split produced a higher Precision@50 of 0.92, but 31 clients were shared between training and testing. The grouped result is therefore a more conservative and relevant measurement for unseen-client evaluation, but it should not be treated as a guaranteed future performance level.

### Claim 3 — Feature interpretation

**Too strong:**
High impressions and search position cause pages to decline.

**Rewritten claim:**
Permutation importance showed that `impressions_90d` and `avg_position` were the strongest measured contributors to the Random Forest ranking on the held-out data.

This represents an association within the model and dataset. It does not establish that these features cause content decline.

### Final public-safe interpretation

The Random Forest provides promising **decision-support** for prioritising pages for human review. On the measured grouped-client test set, it produced a stronger ranked queue than the simple Week-4 baseline.

However, the target is an observed decline proxy and some recent search-performance features may overlap with the proxy measurement period. The results should therefore be treated as **observed and directional evidence**, rather than as proof of future decline or proof that refreshing a recommended page will improve its performance.

A content or SEO specialist should use the ranking as a prioritisation aid and investigate the page before making a refresh decision.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.